In [2]:
import numpy as np

In [3]:
traindata = np.loadtxt("Data_OneStepAhead/Sunspot/train7.txt")
testdata = np.loadtxt("Data_OneStepAhead/Sunspot/test7.txt") 

In [21]:
hidden = 5
input = 7  # max input
output = 1
subtasks = 3
baseNet = [input, hidden, output]
secondnet = [input, hidden+1, output]
thirdnet = [input, hidden+2, output]
mtaskNet = np.array([baseNet, secondnet, thirdnet])


In [5]:
netsize = np.zeros(subtasks, dtype=np.int)
def net_size(netw):
    return ((netw[0] * netw[1]) + (netw[1] * netw[2]) + netw[1] + netw[2])


In [37]:
def decode(w, mtopo, subtasks):
    for n in range(0, subtasks):
        Top = mtopo[n]
    W1 = np.random.randn(Top[0], Top[1] )
    B1 = np.random.randn(Top[1])    # bias first layer
    W2 = np.random.randn(Top[1], Top[2] )
    B2 = np.random.randn(Top[2]) 
    w_layer1size = Top[0] * Top[1]
    w_layer2size =  Top[1] *  Top[2]

    w_layer1 = w[0:w_layer1size]
    W1 = np.reshape(w_layer1, ( Top[0],  Top[1]))

    w_layer2 = w[w_layer1size:w_layer1size + w_layer2size]
    W2 = np.reshape(w_layer2, ( Top[1],  Top[2]))
    B1 = w[w_layer1size + w_layer2size:w_layer1size + w_layer2size +  Top[1]]
    B2 = w[w_layer1size + w_layer2size + Top[1]:w_layer1size + w_layer2size + Top[1] + Top[2]]
    return W1, W2, w

In [38]:
#what happens when we decode?
for n in range(0, subtasks):
    netw = mtaskNet[n]
    netsize[n] =  net_size(netw)
    w = np.random.randn(subtasks, netsize[subtasks-1]) #theta_m = [theta_m-1, psi_m]
#print(w)
#when we use main to put in params for decode:
    print(decode(w[n,:netsize[n]], netw)) 


None
None
None


In [26]:
def decode_MTNencodingX(w, mtopo, subtasks):


    position = 0

    Top1 = mtopo[0]


    for neu in range(0, Top1[1]): #in each neuron of hidden layer
        for row in range(0, Top1[0]):
            W1[row, neu] = w[position] 
            #print neu, row, position, '    -----  a '
            position = position + 1
        B1[neu] = w[position]
        #print neu,   position, '    -----  b '
        position = position + 1

    for neu in range(0, Top1[2]):
        for row in range(0, Top1[1]):
            W2[row, neu] = w[position]
            #print neu, row, position, '    -----  c '
            position = position + 1


    if subtasks >=1:


        #for step  in range(1, subtasks+1   ):
        for step  in range(1, subtasks   ):
            TopPrev = mtopo[step-1] #mtopo = [[Inp, Hidd, Out] of model m-1, [Inp, Hid, Out] of model m]
            TopG = mtopo[step]
            Hid = TopPrev[1]
            Inp = TopPrev[0]


            layer = 0

            for neu in range(Hid , TopG[layer + 1]      ) : #neu from hidden neu of model m-1 to hidden neu of model m
                for row in range(0, TopG[layer]   ): #neu from input layer of model m
                    #print neu, row, position, '    -----  A '

                    W1[row, neu] = w[position] #all the W1 from both model m and m-1 -> we store them into a big list of params w
                    position = position + 1

                B1[neu] = w[position]
                #print neu,   position, '    -----  B '
                position = position + 1

            diff = (TopG[layer + 1] - TopPrev[layer + 1]) # just the diff in number of hidden neurons between subtasks

            for neu in range(0, TopG[layer + 1]- diff):  # % #neurons in hidden layer of model m-1
                for row in range(Inp , TopG[layer]): #neu in input layer of model m-1 to input layer of model m
                    #print neu, row, position, '    -----  C '
                    W1[row, neu] = w[position]
                    position = position + 1

            layer = 1

            for neu in range(0, TopG[layer + 1]):  # %
                for row in range(Hid , TopG[layer]):
                    #print neu, row, position, '    -----  D '
                    W2[row, neu] = w[position]
                    position = position + 1
    return W1, W2, B1, B2, w

In [31]:
decode(w[0,:netsize[0]], mtaskNet[0])

(array([[-1.0628742 , -0.22139486,  0.0070691 , -0.60367268,  0.24269954],
        [-0.56622748, -1.09992967,  0.49407647,  1.16946553, -0.27850964],
        [ 0.06718193,  0.51208644,  1.86169364, -0.11849472,  0.93999124],
        [-1.44252601,  0.77339215,  1.89476573, -1.63738947,  0.82155959],
        [ 0.3974519 , -0.79335267, -0.82987764,  0.36836966,  0.67936477],
        [ 1.05090909,  0.50833628,  0.06684637, -0.22809231,  0.39961805],
        [-1.53629971, -1.05943167,  1.20900538, -0.1804909 ,  0.20897618]]),
 array([[-1.05568405],
        [-0.90576917],
        [ 0.03343664],
        [ 2.00868103],
        [ 0.49130631]]),
 array([-1.0628742 , -0.22139486,  0.0070691 , -0.60367268,  0.24269954,
        -0.56622748, -1.09992967,  0.49407647,  1.16946553, -0.27850964,
         0.06718193,  0.51208644,  1.86169364, -0.11849472,  0.93999124,
        -1.44252601,  0.77339215,  1.89476573, -1.63738947,  0.82155959,
         0.3974519 , -0.79335267, -0.82987764,  0.36836966,  0.6

In [28]:
decode_MTNencodingX(w[0,:netsize[0]], mtaskNet, subtasks)


(array([[-0.01086858,  1.35254007, -1.0987321 , -0.41073998, -1.41556024],
        [ 1.07697601, -0.99692496,  0.24352471,  0.30365934,  1.00571407],
        [-0.82830158, -0.84932898,  1.71362631,  1.14591976,  0.4896236 ],
        [ 0.21375399, -1.0236364 ,  0.03911715, -2.78060476,  0.35297923],
        [ 0.29473498,  0.05745181,  0.40534492, -0.51229065,  0.3054053 ],
        [-0.60079095,  0.11340743,  0.65035948, -0.7113414 ,  0.11536141],
        [ 1.97126752, -0.40017304,  1.15869953, -0.95648737,  1.53484451]]),
 array([[-0.72733943],
        [ 0.95990849],
        [ 1.38701408],
        [-1.27823646],
        [ 0.23188506]]),
 array([-0.43395061,  1.54209099, -1.06337812,  0.86377728, -2.42693826]),
 array([-0.63392024]),
 array([-0.01086858,  1.07697601, -0.82830158,  0.21375399,  0.29473498,
        -0.60079095,  1.97126752, -0.43395061,  1.35254007, -0.99692496,
        -0.84932898, -1.0236364 ,  0.05745181,  0.11340743, -0.40017304,
         1.54209099, -1.0987321 ,  0.24

In [ ]:
W_s1 = w[0,:netsize[0]]
W_s2 = w[0,:netsize[1]]
W_s2

array([-0.88755903,  1.11398156,  1.22550895, -1.58079342, -1.14031121,
       -0.22281787, -0.89373318, -0.58348126,  1.040006  , -1.12931414,
        0.4148672 , -0.59096075,  0.10494975, -0.04455943,  0.67401765,
        0.73831015,  1.48057138, -0.28562645,  1.26651904,  1.32667272,
       -1.56708619, -1.75585671, -1.28206577,  0.30988609, -1.89816032,
        0.28297436, -0.56569446, -1.1743419 ,  0.2734496 , -2.03778415,
        1.17647348,  0.39087229,  0.20459614,  0.13481079,  1.54664403,
       -0.5025249 ,  0.40004334,  0.56770775, -0.6611428 , -0.20172703,
       -1.03132918, -0.0645982 , -0.5256246 ,  1.33008376, -1.46097124,
        0.71176845, -1.66249304, -0.38566259, -2.24264034, -0.90256404,
        0.18280504,  1.32123795, -0.38904132,  0.90014323,  1.17798691])

In [ ]:
shape_diff = np.array(W_s2.shape) - np.array(W_s1.shape)
shape_diff

array([9])

In [ ]:
u = np.random.normal(0, 1, shape_diff[0])
u

array([ 0.85939063,  0.8494059 ,  1.80652272,  1.49506004,  0.71086878,
       -0.8538132 , -0.41170141, -0.51749602, -0.39253533])

In [ ]:
W_s2_prop = np.lib.pad(W_s1, (0, shape_diff[0]), 'constant', constant_values=(0,u))
W_s2_prop

/anaconda3/envs/env_pytorch/lib/python3.6/site-packages/numpy/lib/arraypad.py:490: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray
  x = np.array(x)


array([ 0.15905095,  0.73667485, -0.07379792, -0.13501313,  1.08586443,
        0.03778583, -0.51762687,  1.02656264,  0.55523753, -0.66421731,
       -1.21718198,  1.09457939, -0.48877796,  0.01056669, -1.55126916,
        0.44620071, -0.286809  , -0.81855616, -0.40910699,  0.49779394,
        0.17769591,  1.53499948, -2.38102191,  1.69100163,  0.06862435,
       -0.50186768,  0.36204507,  0.7417608 , -0.31739651, -0.16133594,
        1.64807386,  1.89477947,  0.90736989, -1.05164692,  0.898741  ,
       -1.15217812,  0.62774767, -1.14573687, -0.80067153, -0.62070406,
        0.6886936 ,  1.51825581,  0.13722199, -0.54491557,  0.05336102,
       -1.16414054,  0.85939063,  0.8494059 ,  1.80652272,  1.49506004,
        0.71086878, -0.8538132 , -0.41170141, -0.51749602, -0.39253533])